# Add relationships to IQModel (all 10 gold tables)

Adds the 9 star-schema relationships (fact → dim, many-to-one, single cross-filter) to a manually-created semantic model named **IQModel**, then does the **full refresh** Direct Lake needs so the relationship indexes build.

**Before you run (order matters):**
1. Open this notebook in the **same workspace** as `IQModel` (here: `YOUR_WORKSPACE_NAME`, tenant YOUR_TENANT).
2. **Run the first code cell (`%pip install semantic-link-labs`) by itself FIRST** and wait for its output to show **`Successfully installed … semantic-link-labs-…`**. `sempy_labs` is **not** preinstalled in Fabric — if you skip or interrupt this cell you get `ModuleNotFoundError: No module named 'sempy_labs'`.
3. **Then** run the remaining cells top to bottom. They're idempotent — re-runs skip relationships that already exist or whose tables aren't in the model.

**Relationships added:** fact_transaction → dim_customer / dim_merchant / dim_account / dim_date; fact_fraud_alert → dim_customer; fact_loan → dim_customer; fact_delinquency → dim_customer; dim_account → dim_product / dim_branch.


In [ ]:
%pip install semantic-link-labs

In [ ]:
import sempy_labs as labs
from sempy_labs.tom import connect_semantic_model

DATASET = "IQModel"   # must live in THIS notebook's workspace

# (from_table, from_column, to_table, to_column) - all Many -> One, single direction
RELS = [
    ("fact_transaction",  "customerKey", "dim_customer", "customerKey"),
    ("fact_transaction",  "merchantId",  "dim_merchant", "merchantId"),
    ("fact_transaction",  "accountId",   "dim_account",  "accountId"),
    ("fact_transaction",  "dateKey",     "dim_date",     "dateKey"),
    ("fact_fraud_alert",  "customerKey", "dim_customer", "customerKey"),
    ("fact_loan",         "customerKey", "dim_customer", "customerKey"),
    ("fact_delinquency",  "customerKey", "dim_customer", "customerKey"),
    ("dim_account",       "productId",   "dim_product",  "productId"),
    ("dim_account",       "branchId",    "dim_branch",   "branchId"),
]

with connect_semantic_model(dataset=DATASET, readonly=False) as tom:
    for ft, fc, tt, tc in RELS:
        try:
            tom.add_relationship(
                from_table=ft, from_column=fc,
                to_table=tt, to_column=tc,
                from_cardinality="Many", to_cardinality="One",
            )
            print("added ", ft, fc, "->", tt, tc)
        except Exception as e:
            print("skip  ", ft, fc, ":", e)

In [ ]:
# Direct Lake REQUIRES a full refresh so the new relationship indexes build;
# without it, cross-table measures error with "relationship ... needs to be recalculated".
labs.refresh_semantic_model(dataset=DATASET, refresh_type="full")
print("refreshed")

In [ ]:
# Verify: list the relationships now on the model
import sempy.fabric as fabric
display(fabric.list_relationships(dataset="IQModel"))